In [7]:
import pandas as pd
from scipy.stats import spearmanr, pearsonr

# ========= 1. 读数据 =========
df_yield = pd.read_csv("../../CSV/IAMC/Basin_yield_17regions.csv")
df_moran = pd.read_csv("../../CSV/moran/basin_agri_moran_3f.csv")

# ========= 2. Moran：宽表 → 长表 =========
df_moran_long = df_moran.melt(
    id_vars="year",
    var_name="region_var",
    value_name="moran"
)

df_moran_long["17region"] = (
    df_moran_long["region_var"]
    .str.replace("_basin_agri", "", regex=False)
)

df_moran_long = df_moran_long[["17region", "year", "moran"]]

# ---------- 3. 合并 ----------
df = pd.merge(
    df_moran_long,
    df_yield,
    on=["17region", "year"],
    how="inner"
)

print(df.head())

  17region  year  moran  yield_region
0      BRA  2005  0.826      3.173532
1      BRA  2010  0.831      3.308534
2      BRA  2020  0.832      3.433416
3      BRA  2030  0.827      3.691382
4      BRA  2040  0.814      3.939844


In [8]:
results = []

for reg, d in df.groupby("17region"):
    rho, p = spearmanr(d["yield_region"], d["moran"])
    results.append({
        "region": reg,
        "spearman_rho": rho,
        "p_value": p,
        "n_years": len(d)
    })

df_corr_region = pd.DataFrame(results)

# 按相关系数排序
df_corr_region = df_corr_region.sort_values("spearman_rho")

print(df_corr_region)

   region  spearman_rho       p_value  n_years
8     XAF     -1.000000  0.000000e+00       11
14    XOC     -1.000000  0.000000e+00       11
9    XE25     -1.000000  0.000000e+00       11
3     CIS     -1.000000  0.000000e+00       11
16    XSE     -0.984057  4.666691e-08       11
12    XME     -0.981818  8.403066e-08       11
0     BRA     -0.936364  2.208208e-05       11
5     JPN     -0.931517  3.050778e-05       11
2     CHN     -0.918182  6.661452e-05       11
11    XLM     -0.918182  6.661452e-05       11
4     IND     -0.909091  1.055934e-04       11
7     USA     -0.242569  4.723423e-01       11
10    XER     -0.109091  7.495086e-01       11
6     TUR     -0.068337  8.417603e-01       11
13    XNF      0.036447  9.152761e-01       11
1     CAN      0.045455  8.944270e-01       11
15    XSA      0.519364  1.015732e-01       11


In [9]:
df = df.sort_values(["17region", "year"])

df["d_yield"] = df.groupby("17region")["yield_region"].diff()
df["d_moran"] = df.groupby("17region")["moran"].diff()

df_diff = df.dropna(subset=["d_yield", "d_moran"])

results_diff = []

for reg, d in df_diff.groupby("17region"):
    if len(d) >= 3:
        rho, p = spearmanr(d["d_yield"], d["d_moran"])
        results_diff.append({
            "region": reg,
            "spearman_rho_delta": rho,
            "p_value": p,
            "n_obs": len(d)
        })

df_corr_delta = pd.DataFrame(results_diff)
df_corr_delta = df_corr_delta.sort_values("spearman_rho_delta")

print(df_corr_delta)


   region  spearman_rho_delta   p_value  n_obs
16    XSE           -0.871992  0.001004     10
2     CHN           -0.790277  0.006514     10
14    XOC           -0.624242  0.053718     10
9    XE25           -0.585377  0.075422     10
15    XSA           -0.533783  0.112025     10
8     XAF           -0.518302  0.124838     10
3     CIS           -0.469216  0.171281     10
13    XNF           -0.353665  0.316078     10
1     CAN           -0.310032  0.383320     10
6     TUR           -0.260606  0.467089     10
5     JPN           -0.236364  0.510885     10
7     USA           -0.165145  0.648438     10
10    XER            0.006079  0.986703     10
4     IND            0.135826  0.708302     10
12    XME            0.408544  0.241130     10
0     BRA            0.558324  0.093464     10
11    XLM            0.620064  0.055823     10


In [10]:
df_corr_region.to_csv("../../CSV/corr/corr_yield_moran_by_region.csv", index=False)
df_corr_delta.to_csv("../../CSV/corr/corr_delta_yield_moran_by_region.csv", index=False)